#

#Organização de Dados

O objetivo desse notebook é simplesmente organizar as imagens da pasta de Data para que possamos separar os dados em treino e teste de forma estratégica, já que irei utilizar o pytorch em conjunto com a resnet18 (transfer learning) para resolver esse problema.

In [1]:
import random
import os
import shutil
AI_PATH = r"E:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\Data\raw\Ai_generated_dataset"
REAL_PATH = r"E:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\Data\raw\real_dataset"

In [2]:
def get_all_images(base_path):
    """
    FUNÇÃO PRA LISTAR TODAS AS IMAGENS DEPENDENDO DO DIRETÓRIO (IA OU REAL)
    """
    image_paths = []
    
    for category in os.listdir(base_path):
        category_path = os.path.join(base_path, category)
        
        if os.path.isdir(category_path):
            for img in os.listdir(category_path):
                full_path = os.path.join(category_path, img)
                image_paths.append(full_path)
    
    return image_paths

In [3]:
real_images = get_all_images(REAL_PATH)
ai_images = get_all_images(AI_PATH)

print("Real:", len(real_images))
print("AI:", len(ai_images))

Real: 745
AI: 250


É possível perceber que o dataset tá altamente desbalanceado, temos uma proporção de 3:1 de imagens reais do que geradas por inteligência artificial. Por esse motivo eu irei colocar peso nas classes (Class Weights), com o intuito de que treinar o modelo pensando que errar IA é mais grave do que errar real.
peso_classe = total_amostras / (n_classes * amostras_da_classe)

In [4]:
total = len(real_images)+len(ai_images)
n_classes = 2

In [5]:
peso_real = total / (n_classes * len(real_images))
peso_ai = total / (n_classes * len(ai_images))
print(f"Peso classe IA: {peso_ai} | Peso classe Real: {peso_real}")

Peso classe IA: 1.99 | Peso classe Real: 0.6677852348993288


Como vamos utilizar pytorch, vou precisar converter esses pesos em tensor. 

Classe 0 - AI

Classe 1 - REAL

In [7]:
import torch 

weights = torch.tensor([peso_ai, peso_real])
weights

tensor([1.9900, 0.6678])

##Organizando as imagens nas pastas de teste, treino e validação

In [8]:
BASE_DIR = r"E:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image"

In [26]:
def copy_images(image_list, split_name, label):
    """
    FUNÇÃO PARA COPIAR AS IMAGENS DENTRO DAS RESPECTIVAS PASTAS
    """
    
    for img_path in image_list:
        category = os.path.basename(os.path.dirname(img_path))
        
        file_name = f"{category}_{os.path.basename(img_path)}"
        
        destination = os.path.join(
        BASE_DIR,
        "Data",
        split_name,
        label,
        file_name
        )
        
        shutil.copy(img_path, destination)


In [27]:
import random

def split_data(image_list, train_ratio=0.7, val_ratio=0.15):
    """
    FUNÇÃO PARA DIVIDIR DADOS DE FORMA ALEATÓRIA A PARTIR DE SUA CLASSE
    """
    random.shuffle(image_list)
    
    total = len(image_list)
    train_end = int(total * train_ratio)
    val_end = int(total * (train_ratio + val_ratio))
    
    train = image_list[:train_end]
    val = image_list[train_end:val_end]
    test = image_list[val_end:]
    
    return train, val, test

In [28]:
real_train, real_val, real_test = split_data(real_images)
ai_train, ai_val, ai_test = split_data(ai_images)

In [29]:
copy_images(real_train, "train", "real")
copy_images(real_val, "val", "real")
copy_images(real_test, "test", "real")

copy_images(ai_train, "train", "ai")
copy_images(ai_val, "val", "ai")
copy_images(ai_test, "test", "ai")

Implementei split estratificado por classe para evitar data leakage e garantir distribuição consistente!

In [9]:
TRAIN_DIR = r"E:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\Data\train"
VAL_DIR = r"E:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\Data\val"
TEST_DIR = r"E:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\Data\test"

In [10]:
train_images = get_all_images(TRAIN_DIR)
val_images = get_all_images(VAL_DIR)
test_images = get_all_images(TEST_DIR)
#printando o número de imagens de cada diretório
print("Treino: ", len(train_images))
print("Validação: ", len(val_images))
print("Teste: ", len(test_images))

Treino:  696
Validação:  149
Teste:  150


In [11]:
from torchvision import datasets

In [12]:
train_dataset = datasets.ImageFolder(root=TRAIN_DIR)
val_dataset = datasets.ImageFolder(root = VAL_DIR)
test_dataset = datasets.ImageFolder(root = TEST_DIR)

O ImageFolder organiza automaticamente imagens em labels a partir da estrutura de diretórios, mas não realiza split de dados,essa etapa eu implementei manualmente para evitar data leakage.

In [13]:
print(train_dataset.classes)
print(train_dataset.class_to_idx)
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))
print(weights)
#verificando o label que o ImageFolder deu para cada uma das duas classes e se está tudo certo com o número de imagens!

['ai', 'real']
{'ai': 0, 'real': 1}
696
149
150
tensor([1.9900, 0.6678])


#Transforms (Data Augmentation)

Realizando modificações nas imagens de treino com transforms para uma melhor validação do modelo.

In [14]:
from torchvision import transforms
import torch 

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    
    transforms.ToTensor(),
    
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])





🧠 O que está acontecendo aqui:

Resize → padrão do modelo

Augmentation → robustez

ToTensor → converte e escala (0–1)

Normalize → padrão ImageNet

TRANSFORM DE VALIDAÇÃO/TESTE

In [15]:
transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    
    transforms.ToTensor(),
    
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

Conectando com o Dataset

In [16]:
train_dataset = datasets.ImageFolder(root=TRAIN_DIR,transform=transform_train)
val_dataset = datasets.ImageFolder(root = VAL_DIR,transform=transform_eval)
test_dataset = datasets.ImageFolder(root = TEST_DIR, transform=transform_eval)

In [17]:
img, label = train_dataset[0]

print(img.shape)
print(label)

torch.Size([3, 224, 224])
0


#Implementando o DataLoader

In [18]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset= train_dataset,
    batch_size= 32,
    shuffle = True
)

val_loader = DataLoader(
    dataset= val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    dataset= test_dataset,
    batch_size= 32,
    shuffle= False
)

Testando se o loader funcionou

In [19]:
for images, labels in train_loader:
    print(images.shape)
    print(labels.shape)
    break

torch.Size([32, 3, 224, 224])
torch.Size([32])


#Carregando a RESNET18

In [20]:
import torchvision.models as models
import torch.nn as nn 
model = models.resnet18(pretrained = True)
print(model)


e:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

Eu congelo as camadas iniciais porque elas já aprenderam features gerais (bordas, texturas, formas) no ImageNet. Assim, evito overfitting e reduzo custo computacional, focando apenas na adaptação da camada final ao meu problema específico.

In [21]:
for param in model.parameters(): #congelando todas as camadas
    param.requires_grad = False
    
model.fc = torch.nn.Linear(model.fc.in_features, 2) #cria nova camada que é treinável por padrão

In [22]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

fc.weight
fc.bias


O problema tem duas saídas, vai ser natural para essa classificação multiclasse o uso de uma loss function como a CrossEntropyLoss. Como otimizador, irei utilizar o Adam. Utilizei uma função lambda para garatir que estou filtrando apenas a camada final para o treino.

In [23]:
loss_fn = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

#Treinando o modelo

Irei também definir o device por aqui (CPU ou GPU). Esse é um procedimento padrão, nesse caso para treinar com a GPU eu precisaria utilizar um ambiente como o google colab!

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


 Devido ao tamanho reduzido do dataset e ao uso de transfer learning com camadas congeladas, o treinamento será realizado com a CPU sem impacto significativo no tempo de execução. Em cenários maiores, o uso de GPU seria essencial para eficiência. Estou rodando o pytorch localmente e não tenho uma gpu da NVIDIA, a minha é da AMD (RX 7800XT).

seguindo a ordem de treino do pytorch:
1. zero_grad()
2. forward
3. loss
4. backward()
5. optimizer.step()

In [28]:
epochs = 10

for epoch in range(epochs):
    
    # ===== TREINO =====
    model.train()
    train_loss = 0
    
    for images, labels in train_loader:
        
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    
    
    # ===== VALIDAÇÃO =====
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            
            val_loss += loss.item()
            
            _, preds = torch.max(outputs, 1)
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    val_loss /= len(val_loader)
    accuracy = correct / total
    
    
    # ===== LOG =====
    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val Accuracy: {accuracy:.4f}")
    print("-" * 30)
    
        

Epoch 1/10
Train Loss: 0.2322
Val Loss: 0.2769
Val Accuracy: 0.8993
------------------------------
Epoch 2/10
Train Loss: 0.2154
Val Loss: 0.2713
Val Accuracy: 0.8926
------------------------------
Epoch 3/10
Train Loss: 0.2267
Val Loss: 0.3054
Val Accuracy: 0.8926
------------------------------
Epoch 4/10
Train Loss: 0.2065
Val Loss: 0.3612
Val Accuracy: 0.8725
------------------------------
Epoch 5/10
Train Loss: 0.1965
Val Loss: 0.3145
Val Accuracy: 0.8926
------------------------------
Epoch 6/10
Train Loss: 0.1695
Val Loss: 0.3035
Val Accuracy: 0.9060
------------------------------
Epoch 7/10
Train Loss: 0.1950
Val Loss: 0.2775
Val Accuracy: 0.9060
------------------------------
Epoch 8/10
Train Loss: 0.1958
Val Loss: 0.2745
Val Accuracy: 0.9128
------------------------------
Epoch 9/10
Train Loss: 0.2162
Val Loss: 0.2872
Val Accuracy: 0.9060
------------------------------
Epoch 10/10
Train Loss: 0.1746
Val Loss: 0.2790
Val Accuracy: 0.8993
------------------------------


In [30]:
from sklearn.metrics import f1_score

all_preds = []
all_labels = []

test_loss = 0
correct = 0
total = 0

model.to(device)
model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        
        # mover para device
        images = images.to(device)
        labels = labels.to(device)
        
        # forward
        outputs = model(images)
        
        # loss
        loss = loss_fn(outputs, labels)
        test_loss += loss.item()
        
        # predições
        _, preds = torch.max(outputs, 1)
        
        # salvar preds e labels
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        # accuracy
        correct += (preds == labels).sum().item()
        total += labels.size(0)

# média da loss
test_loss /= len(test_loader)

# accuracy final
accuracy = correct / total

# F1-score
f1 = f1_score(all_labels, all_preds, pos_label = 0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"F1-score: {f1:.4f}")

Test Loss: 0.2182
Test Accuracy: 0.9200
F1-score: 0.8500


Considerando que tivemos um dataset pequeno, utilizei a CPU para rodar o treino e fizemos apenas o fine-tuning da última camada do modelo, esse resultado está satisfatório. A avaliação foi ajustada para priorizar a detecção de imagens geradas por IA, tratando essa classe como positiva no cálculo do F1-score

#Matriz de confusão

In [31]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds)

print(cm)

[[ 34   4]
 [  8 104]]


## 📊 Avaliação do Modelo

### Matriz de Confusão

O modelo foi avaliado em um conjunto de teste separado, resultando na seguinte matriz de confusão:

|               | Predito IA | Predito Real |
|--------------|------------|--------------|
| **Real (IA)**   | 34         | 4            |
| **Real (Real)** | 8          | 104          |

---

### 🎯 Principais Insights

- O modelo identificou corretamente **34 de 38 imagens geradas por IA**, atingindo um **recall de ~89%** para essa classe.
- Foram classificadas incorretamente **4 imagens de IA como reais**, representando o tipo de erro mais crítico nesse contexto.
- Além disso, **8 imagens reais foram classificadas como IA**, indicando uma certa sensibilidade a padrões visuais semelhantes aos gerados artificialmente.

---

### ⚖️ Trade-off entre Precisão e Recall

- **Precisão (classe IA): ~81%**  
  → Quando o modelo prevê que uma imagem é gerada por IA, ele está correto em cerca de 81% dos casos.

- **Recall (classe IA): ~89%**  
  → O modelo consegue identificar a maioria das imagens geradas por IA.

Isso indica que o modelo prioriza a **detecção de conteúdo gerado por IA**, mesmo que isso resulte em alguns falsos positivos.

---

### 🧠 Interpretação

Do ponto de vista prático, esse comportamento é desejável em cenários onde:

- Deixar passar uma imagem gerada por IA é mais crítico do que classificar uma imagem real como IA
- O sistema atua como um **filtro ou ferramenta de triagem inicial**

---

### 🚀 Oportunidades de Melhoria

Para melhorar ainda mais o desempenho (principalmente a precisão), algumas abordagens podem ser exploradas:

- Realizar fine-tuning de camadas mais profundas do modelo (além da camada final)
- Aumentar o uso de data augmentation para melhorar a generalização
- Ajustar o threshold de decisão para equilibrar precisão e recall
- Expandir o dataset para reduzir variância

---

### 📌 Conclusão

O modelo apresenta um bom desempenho considerando o tamanho do dataset e o uso de transfer learning, atingindo:

- **~92% de acurácia**
- **~85% de F1-score (classe IA)**

mantendo um alto recall na detecção de imagens geradas por IA.

#Salvando o modelo


In [32]:
torch.save(model.state_dict(), "model.pth")

In [37]:
#Carregando o modelo
MODEL_PATH = r"E:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\model\model.pth"
model = models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(model.fc.in_features, 2)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()

e:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [40]:
from PIL import Image
import torchvision.transforms as transforms
import torch

def predict_image(image_path, model, device):
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0)
    
    image = image.to(device)
    
    model.eval()
    
    with torch.no_grad():
        outputs = model(image)
        probs = torch.softmax(outputs, dim=1)
        _, pred = torch.max(outputs, 1)
        confidence = probs[0][pred].item()
    
    classes = ["AI", "Real"]
    
    return classes[pred.item()], confidence

#Testando o modelo com 2 imagens diferentes 

In [47]:
#gerada por IA
path_image_1 = r"E:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\test_images\fight.jpg"
label, conf = predict_image(path_image_1, model, device)

print(f"Predição: {label} | Confiança: {conf:.2f}")

Predição: Real | Confiança: 0.63


In [ ]:
#REAL
path_image_2 = r"E:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\test_images\fight2.jpg"
label, conf = predict_image(path_image_2, model, device)

print(f"Predição: {label} | Confiança: {conf:.2f}")

Predição: Real | Confiança: 0.90


In [53]:
# IA 2
path_image_3 = r"E:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\test_images\fight3.jpg"
label, conf = predict_image(path_image_3, model, device)

print(f"Predição: {label} | Confiança: {conf:.2f}")

Predição: Real | Confiança: 0.70


In [54]:
path_image_4 = r"E:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\test_images\fight4.jpg"
label, conf = predict_image(path_image_4, model, device)

print(f"Predição: {label} | Confiança: {conf:.2f}")

Predição: Real | Confiança: 0.68
